# Pokemon Card Extractor & Whatnot Search

Extract structured data from Pokemon card images using OpenAI's structured outputs (`response_format`), then search Whatnot.

## 1. Setup & Install

In [21]:
!pip install openai python-dotenv pydantic pandas --quiet

In [22]:
import os
import base64
import webbrowser
from urllib.parse import quote
from typing import Optional, List

import openai
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import display, Image, Markdown

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

## 2. Data Model (Pydantic)

This is the **single source of truth** for what we extract from every card.  
OpenAI's `response_format` guarantees the response conforms to this schema — no JSON parsing, no cleanup, no code fences to strip.

In [23]:
class PokemonCard(BaseModel):
    """Structured representation of a Pokemon card extracted from an image."""

    pokemon_name: str = Field(description="Name of the Pokemon (e.g. 'Mewtwo', 'Charizard ex')")
    hp: Optional[int] = Field(default=None, description="Hit points shown on the card")
    card_number: Optional[str] = Field(default=None, description="Card number (e.g. '150/165')")
    set_name: Optional[str] = Field(default=None, description="Set name (e.g. 'Scarlet & Violet 151')")
    series: Optional[str] = Field(default=None, description="Series (e.g. 'Scarlet & Violet', 'Sword & Shield')")
    edition: Optional[str] = Field(default=None, description="Edition (e.g. '1st Edition', 'Unlimited')")
    rarity: Optional[str] = Field(default=None, description="Rarity (e.g. 'Holo Rare', 'Ultra Rare', 'Common')")
    illustrator: Optional[str] = Field(default=None, description="Artist/illustrator credited on the card")
    year: Optional[int] = Field(default=None, description="Copyright year printed on the card")
    card_type: Optional[str] = Field(default=None, description="Card type (e.g. 'Basic', 'Stage 1', 'V', 'VMAX', 'ex')")
    weakness: Optional[str] = Field(default=None, description="Weakness type and modifier")
    resistance: Optional[str] = Field(default=None, description="Resistance type and modifier")
    retreat_cost: Optional[int] = Field(default=None, description="Number of energy symbols for retreat cost")
    attacks: List[str] = Field(default_factory=list, description="List of attack names on the card")
    additional_notes: Optional[str] = Field(default=None, description="Promo info, special markings, errors, etc.")

    @property
    def display_name(self) -> str:
        parts = [self.pokemon_name]
        if self.card_number:
            parts.append(f"({self.card_number})")
        if self.set_name:
            parts.append(f"- {self.set_name}")
        if self.rarity:
            parts.append(f"- {self.rarity}")
        return " ".join(parts)

    @property
    def whatnot_search_query(self) -> str:
        """Build a focused Whatnot search string from the structured fields."""
        tokens = ["Pokemon card", self.pokemon_name]
        if self.card_number:
            tokens.append(self.card_number)
        if self.set_name:
            tokens.append(self.set_name)
        if self.edition:
            tokens.append(self.edition)
        if self.rarity:
            tokens.append(self.rarity)
        return " ".join(tokens)

    @property
    def whatnot_url(self) -> str:
        return f"https://www.whatnot.com/search?query={quote(self.whatnot_search_query)}&referringSource=typed"

    def to_markdown_table(self) -> str:
        """Render card data as a markdown table for Jupyter display."""
        lines = ["| **Field** | **Value** |", "| :-- | :-- |"]
        for field_name, value in self.model_dump().items():
            if value is not None and value != []:
                label = field_name.replace("_", " ").title()
                if isinstance(value, list):
                    value = ", ".join(value)
                lines.append(f"| {label} | {value} |")
        return "\n".join(lines)

    def _repr_markdown_(self):
        """Rich display in Jupyter notebooks."""
        return f"### {self.display_name}\n\n{self.to_markdown_table()}"

## 3. Extraction Function

Uses `openai.beta.chat.completions.parse()` with `response_format=PokemonCard`.  
The API returns a **parsed Pydantic object** directly — no JSON wrangling needed.

In [24]:
SYSTEM_PROMPT = (
    "You are an expert Pokemon card analyst. Given an image of a Pokemon card, "
    "extract every piece of structured information you can identify from the card. "
    "Be precise with card numbers, set names, and illustrator credits. "
    "Use null for any field you cannot confidently determine from the image."
)


def build_image_content(image_source: str) -> dict:
    """Build the image content block for either a URL or a local file path."""
    if image_source.startswith(("http://", "https://")):
        return {
            "type": "image_url",
            "image_url": {"url": image_source},
        }
    else:
        with open(image_source, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        ext = os.path.splitext(image_source)[1].lower().lstrip(".")
        mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "webp": "webp"}.get(ext, "jpeg")
        return {
            "type": "image_url",
            "image_url": {"url": f"data:image/{mime};base64,{b64}"},
        }


def extract_card(image_source: str) -> PokemonCard:
    """
    Send an image to OpenAI's vision model and return a validated PokemonCard.

    Uses structured outputs (response_format) so the API returns a
    Pydantic-validated object directly — no JSON parsing required.
    """
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Extract all structured data from this Pokemon card image."},
                    build_image_content(image_source),
                ],
            },
        ],
        response_format=PokemonCard,
        temperature=0.1,
    )

    card = completion.choices[0].message.parsed

    if card is None:
        refusal = completion.choices[0].message.refusal
        raise ValueError(f"Extraction failed: {refusal or 'No structured output returned'}")

    return card

## 4. Extract a Card!

Change the URL below to any Pokemon card image.

In [25]:
IMAGE_URL = "https://m.media-amazon.com/images/I/71nbfl-JklS._AC_SY606_.jpg"

# Preview the image inline
display(Image(url=IMAGE_URL, width=300))

In [26]:
card = extract_card(IMAGE_URL)

# Rich table display via _repr_markdown_
card

### Charizard (11/108) - XY Evolutions - Holo Rare

| **Field** | **Value** |
| :-- | :-- |
| Pokemon Name | Charizard |
| Hp | 150 |
| Card Number | 11/108 |
| Set Name | XY Evolutions |
| Series | XY |
| Edition | null |
| Rarity | Holo Rare |
| Illustrator | Mitsuhiro Arita |
| Year | 2016 |
| Card Type | Stage 2 |
| Weakness | Water ×2 |
| Resistance | Fighting -20 |
| Retreat Cost | 3 |
| Attacks | Fire Spin |
| Additional Notes | Ability: Energy Burn. All Energy attached to this Pokémon are Energy instead of their usual type. |

## 5. Inspect the Data

Since it's a Pydantic model, you get typed access to every field.

In [27]:
# Access fields directly — fully typed, IDE-friendly
print(f"Pokemon: {card.pokemon_name}")
print(f"HP: {card.hp}")
print(f"Illustrator: {card.illustrator}")
print(f"Year: {card.year}")
print(f"Set: {card.set_name}")
print(f"Attacks: {card.attacks}")

Pokemon: Charizard
HP: 150
Illustrator: Mitsuhiro Arita
Year: 2016
Set: XY Evolutions
Attacks: ['Fire Spin']


In [28]:
# Dump to dict or JSON — ready for storage, APIs, whatever
card.model_dump()

{'pokemon_name': 'Charizard',
 'hp': 150,
 'card_number': '11/108',
 'set_name': 'XY Evolutions',
 'series': 'XY',
 'edition': 'null',
 'rarity': 'Holo Rare',
 'illustrator': 'Mitsuhiro Arita',
 'year': 2016,
 'card_type': 'Stage 2',
 'weakness': 'Water ×2',
 'resistance': 'Fighting -20',
 'retreat_cost': 3,
 'attacks': ['Fire Spin'],
 'additional_notes': 'Ability: Energy Burn. All Energy attached to this Pokémon are Energy instead of their usual type.'}

In [29]:
# Or JSON string
print(card.model_dump_json(indent=2))

{
  "pokemon_name": "Charizard",
  "hp": 150,
  "card_number": "11/108",
  "set_name": "XY Evolutions",
  "series": "XY",
  "edition": "null",
  "rarity": "Holo Rare",
  "illustrator": "Mitsuhiro Arita",
  "year": 2016,
  "card_type": "Stage 2",
  "weakness": "Water ×2",
  "resistance": "Fighting -20",
  "retreat_cost": 3,
  "attacks": [
    "Fire Spin"
  ],
  "additional_notes": "Ability: Energy Burn. All Energy attached to this Pokémon are Energy instead of their usual type."
}


## 6. Whatnot Search

In [30]:
print(f"Search query: {card.whatnot_search_query}")
display(Markdown(f"**[Search Whatnot for this card]({card.whatnot_url})**"))

Search query: Pokemon card Charizard 11/108 XY Evolutions null Holo Rare


**[Search Whatnot for this card](https://www.whatnot.com/search?query=Pokemon%20card%20Charizard%2011/108%20XY%20Evolutions%20null%20Holo%20Rare&referringSource=typed)**

In [31]:
# Uncomment to open in your browser automatically
# webbrowser.open(card.whatnot_url)

## 7. Batch Processing

Process multiple cards → pandas DataFrame in one shot.

In [32]:
urls = [
    "https://m.media-amazon.com/images/I/71nbfl-JklS._AC_SY606_.jpg",
    "https://tcgplayer-cdn.tcgplayer.com/product/246881_in_1000x1000.jpg",
    "https://m.media-amazon.com/images/I/514L7XZClXL._AC_UF894,1000_QL80_.jpg"
]

cards: list[PokemonCard] = []
for url in urls:
    try:
        c = extract_card(url)
        cards.append(c)
        print(f"✓ {c.display_name}")
    except Exception as e:
        print(f"✗ Failed on {url}: {e}")

print(f"\nExtracted {len(cards)} card(s)")

✓ Charizard (11/108) - XY Evolutions - Holo Rare
✓ Pikachu (049/203) - null - null
✓ Pikachu V (043/185)

Extracted 3 card(s)


In [33]:
# Pydantic → DataFrame in one line
df = pd.DataFrame([c.model_dump() for c in cards])

# Add computed columns
df["whatnot_query"] = [c.whatnot_search_query for c in cards]
df["whatnot_url"] = [c.whatnot_url for c in cards]

display(df)

,pokemon_name,hp,card_number,set_name,series,edition,rarity,illustrator,year,card_type,weakness,resistance,retreat_cost,attacks,additional_notes,whatnot_query,whatnot_url
0,Charizard,150,11/108,XY Evolutions,XY,null,Holo Rare,Mitsuhiro Arita,2016,Stage 2,Water ×2,Fighting -20,3.0,[Fire Spin],Ability: Energy Burn. All Energy attached to t...,Pokemon card Charizard 11/108 XY Evolutions nu...,https://www.whatnot.com/search?query=Pokemon%2...
1,Pikachu,60,049/203,null,null,E,null,chibi,2021,Basic,×2,null,NaN,"[Energize, Electro Ball]","When Pikachu meet, they'll touch their tails t...",Pokemon card Pikachu 049/203 null E null,https://www.whatnot.com/search?query=Pokemon%2...
2,Pikachu V,190,043/185,None,None,None,None,Saki Hayashiro,2020,Basic,×2,None,NaN,"[Charge, Thunderbolt]","V rule: When your Pokémon V is Knocked Out, yo...",Pokemon card Pikachu V 043/185,https://www.whatnot.com/search?query=Pokemon%2...
